In [ ]:
from pathlib import Path
import asyncio
import logging
import re
from urllib.parse import urlparse

import dask.bag as db
from dask.diagnostics import ProgressBar
from dask.distributed import Client, LocalCluster, WorkerPlugin, get_worker
from dask.distributed import as_completed
from coiled import Cluster as CoiledCluster

from obstore.store import S3Store
import rustac
import duckdb


logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
%%time

year_file_re = re.compile(r'.*/(\d{4})\.ndjson$')

destination = Path("./data/geoparquet")
source_store = S3Store(
    bucket="its-live-data", prefix="test-space/stac_catalogs/", region="us-west-2", skip_signature=True
)

paths = []
sizes = []
for list_stream in source_store.list():
    for object_meta in list_stream:
        if year_file_re.match(object_meta["path"]):
            paths.append(object_meta["path"])
            sizes.append(object_meta["size"])
print(len(paths))

In [ ]:
class RustacStorePlugin(WorkerPlugin):
    def __init__(self, store_config):
        self.store_config = store_config
        self.store = None
    
    def setup(self, worker):
        """Initialize the store when a worker starts"""
        bucket,prefix,region = self.store_config
        self.store = S3Store(bucket=bucket, prefix=prefix, region=region, skip_signature=True)
        print("Initialized rustac store")
    
    def teardown(self, worker):
        """Cleanup when worker shuts down"""
        if self.store:
            # Add any store cleanup logic here
            print("Cleaned up rustac store")
    
    def get_store(self):
        if self.store is None:
            raise RuntimeError("Store not initialized")
        return self.store

def create_dask_cluster(environment="local",
                        obstore_opts=None,
                        n_workers=4,
                        threads_per_worker=1,
                        cloud_opts={}):

    
    if "client" in locals() and "cluster" in locals():
        return (client, cluster)
    else:
        if environment=="local":
            print("Creating new local Dask client")
            cluster = LocalCluster(n_workers=n_workers,
                                 threads_per_worker=threads_per_worker,
                                 silence_logs=logging.ERROR)
        else:
            print("Creating new Coiled Dask client")
            cluster = CoiledCluster(n_workers=n_workers, 
                                    threads_per_worker=threads_per_worker,
                                    **cloud_opts)    

        client = Client(cluster)
        obstore_plugin = RustacStorePlugin(obstore_opts)
        client.register_plugin(obstore_plugin, name='rustac_store')
        return (client, cluster)

In [ ]:
store_conf = (
    "its-live-data",
    "test-space/stac_catalogs/landsatOLI/v02",
    "us-west-2"
)

client, cluster = create_dask_cluster(
    environment="local",
    obstore_opts=store_conf,                                  
    n_workers=8,
    threads_per_worker=1)
client

In [ ]:

async def async_process_batch(paths_batch, destination, semaphore_limit=10):
    """Process a batch of paths with async concurrency within the batch"""
    semaphore = asyncio.Semaphore(semaphore_limit)
    
    async def process_single_path(path):
        async with semaphore:
            try:

                if path.startswith(('s3://', 'http://')):
                    source_path = urlparse(path).path
                else:
                    source_path = Path(path)
                
                preserved_path = Path(*url_path.parts[-5:])
                
                # Create destination path with .parquet extension
                destination_path = Path(destination) / preserved_path.with_suffix(".parquet")
                print(destination_path, preserved_path)
                                
                if destination_path.exists():
                    return {'status': 'skipped', 'path': path}
                    
                # Create the directory if it doesn't exist
                destination_path.parent.mkdir(parents=True, exist_ok=True)
                
                # Get the plugin from the worker
                worker = get_worker()
                plugin = worker.plugins['rustac_store']
                store = plugin.store
                
                value = await rustac.read(str(path), store=store)
                
                if value["type"] == "Feature":
                    await rustac.write(str(destination_path), [value])
                else:
                    assert value["type"] == "FeatureCollection"
                    await rustac.write(str(destination_path), value)
                
                return {'status': 'success', 'path': path}
                
            except Exception as e:
                logger.error(f"Failed to process {path}: {str(e)}")
                return {'status': 'failed', 'path': path, 'error': str(e)}
    
    # Process all paths in this batch concurrently
    tasks = [process_single_path(path) for path in paths_batch]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Handle any exceptions that were returned
    processed_results = []
    for result in results:
        if isinstance(result, Exception):
            processed_results.append({
                'status': 'failed', 
                'path': 'unknown', 
                'error': str(result)
            })
        else:
            processed_results.append(result)
    
    return processed_results

def run_async_batch(paths_batch, destination, semaphore_limit=10):
    """Wrapper to run async batch processing in a worker"""
    return asyncio.run(async_process_batch(paths_batch, destination, semaphore_limit))

def process_files_with_async_batches(client, paths, destination, batch_size=50, semaphore_limit=10, progress=True):
    """
    Process files using Dask workers, with async concurrency within each batch
    
    Architecture:
    - Dask distributes batches across workers (e.g., 4 workers get 4 different batches)
    - Each worker creates ONE event loop and processes its batch of paths concurrently
    - Within each batch, semaphore controls concurrent async operations
    
    Parameters:
    - batch_size: Number of paths per worker (e.g., 50 means each worker gets 50 paths)
    - semaphore_limit: Max concurrent async operations within each worker's batch
    """
    
    # Split paths into batches for distribution across workers
    batches = [paths[i:i + batch_size] for i in range(0, len(paths), batch_size)]
    
    print(f"Processing {len(paths)} paths in {len(batches)} batches")
    print(f"Each worker will process up to {batch_size} paths concurrently (limited by semaphore: {semaphore_limit})")
    
    # Submit each batch to a worker - each worker gets ONE batch and creates ONE event loop
    futures = [
        client.submit(run_async_batch, batch, destination, semaphore_limit)
        for batch in batches
    ]
    
    if progress:
        import tqdm
        all_results = []
        with tqdm.tqdm(total=len(paths), desc="Processing files") as pbar:
            for future in as_completed(futures):
                batch_results = future.result()
                all_results.extend(batch_results)
                pbar.update(len(batch_results))
        return all_results
    else:
        batch_results = client.gather(futures)
        # Flatten the results
        return [result for batch in batch_results for result in batch]


results = process_files_with_async_batches(
    client, 
    paths, 
    destination, 
    batch_size=400,  # Paths per worker
    semaphore_limit=20,  # Concurrent async ops per worker
    progress=False
)

In [ ]:
# Debug what's happening with path parts
from pathlib import Path
from urllib.parse import urlparse

# Example path
path = "s3://its-live-data/test-space/stac-catalog/landsatOLI/v02/N30E100/2013.ndjson"

url_path = urlparse(path).path
print(f"url_path: {url_path}")

source_path = Path(url_path)
print(f"source_path.parts: {source_path.parts}")
print(f"Last 4 parts: {source_path.parts[-4:]}")

preserved_path = Path(*source_path.parts[-4:])
print(f"preserved_path: {preserved_path}")

# The fix - skip the root '/' by filtering it out
non_root_parts = [part for part in source_path.parts if part != '/']
print(f"non_root_parts: {non_root_parts}")
print(f"Last 4 non-root parts: {non_root_parts[-4:]}")

preserved_path_fixed = Path(*non_root_parts[-4:])
print(f"preserved_path_fixed: {preserved_path_fixed}")

In [ ]:


paths = [
    'landsatOLI/v02/N20E080/1987.ndjson',
    'landsatOLI/v02/N20E080/',
    'landsatOLI/v02/N20E080/README.txt',
    'landsatOLI/v02/S10W100/2003.ndjson'
]

filtered = [p for p in paths if year_file_re.match(p)]
print(filtered)